In [ ]:
from pathlib import Path

import holoviews as hv
import hvplot.pandas  # noqa+
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from loguru import logger

from eicu_db_connection import DatabaseConnection

hv.extension("bokeh")

%matplotlib inline

In [ ]:
entire_dataset = True

In [ ]:
notebook_name = "vitalaperiodic_exploration"

plot_outputs_dir_small = "subset"
plot_outputs_dir_all = "all"

INTER_DATA_PATH = "../intermediary_data/"
print(f"Loading and saving data from/to {Path(INTER_DATA_PATH).resolve()}")

In [ ]:
connection = DatabaseConnection()
engine = connection.engine
query_schema = connection.query_schema

In [ ]:
conn = engine.connect().execution_options(
        stream_results=True)

In [ ]:
apache_results_valid_filename = "apache_aps_var_results_valid_mortality.csv"
apache_results_valid_path = Path(INTER_DATA_PATH) / Path(apache_results_valid_filename) 

df_apache_valid = pd.read_csv(filepath_or_buffer=apache_results_valid_path)
df_apache_valid = df_apache_valid.drop(columns=["Unnamed: 0", "Unnamed: 0_x", "Unnamed: 0_y"])
df_apache_valid.head(n=3)

In [ ]:
patient_stays_valid: list = df_apache_valid["patientunitstayid"].tolist()

In [ ]:
query_valid_apache = query_schema + f"""
SELECT * 
FROM vitalaperiodic
WHERE patientunitstayid in ({", ".join(map(str, patient_stays_valid))}) 
ORDER BY patientunitstayid, observationoffset
LIMIT 50000
"""

### Getting the vitalaperiodic data for patients with valid apache mortality prediction

In [ ]:
aperiodic_path = Path(INTER_DATA_PATH) / Path("vital_periodic_with_apache_50000.csv")
#vp_apache_path = Path(INTER_DATA_PATH) / Path("vital_periodic_with_apache_all.csv")

#if vp_apache_path.is_file():
#    df_vp_apache_valid = pd.read_csv(vp_apache_path, engine="pyarrow")
#    logger.info(f"Loading data from local csv.")
#else:
#    df_vp_apache_valid = pd.read_sql_query(query_valid_apache, engine)
#    df_vp_apache_valid.to_csv(path_or_buf=vp_apache_path)
#    logger.info(f"Loading data from database.")